# Chapter 19 — Multi-Agent Consensus and Cross-Document Consistency

This notebook accompanies **Chapter 19** of *How to Test and Validate Agentic AI Systems*.

**Learning objectives:**
- Compute inter-agent tension for multi-agent systems
- Detect cross-document factual conflicts
- Use geometric primitives for consensus measurement

**Modules:** `harness.doe.multi_agent`, `harness.doe.cross_document`

In [ ]:
# === Preamble (all notebooks) ===
import sys
import os
import warnings
warnings.filterwarnings("ignore")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# [AUTO-INJECTED] Disable DOE tracing for parallel execution safety
from knowlytix.harness.doe import DOEHarnessConfig
DOEHarnessConfig.__dataclass_fields__['enable_tracing'].default = False


In [ ]:
from importlib.resources import files
# Load the primary store (Chapter 4)
from knowlytix.harness.doe import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)
bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"\nStore: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

In [ ]:
from importlib.resources import files
# Create a second store with different training (100 epochs)
# to simulate a second agent's independent perspective
config2 = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch19_second_perspective",
    n_runs=4,
    ingest_epochs=100,
    device=str(DEVICE),
)
bench2 = DOEGMSBenchmark(config2)
store2 = bench2.ingest()
print(f"\nStore 2: {store2.stats()['entities']} entities, "
      f"{store2.stats()['triples']} triples, "
      f"{store2.stats()['enm_entries']} ENM")

## Agent Roles and Mock Agents

GMSH defines agent roles via `AgentRole` and creates mock agents
for each role with `create_agent_for_role()`.

In [ ]:
# Define agent roles
from knowlytix.harness.doe.multi_agent import (
    AgentRole, InteractionResult,
    ConsensusOracle, MultiAgentGenerator,
)
from knowlytix.harness.fixtures import create_agent_for_role

role_analyst = AgentRole(
    role_id="analyst",
    system_prompt="You are a credit risk analyst. Focus on model "
                  "performance metrics and data quality.",
    allowed_tools=["data_lookup", "risk_check"],
)
role_reviewer = AgentRole(
    role_id="reviewer",
    system_prompt="You are a compliance reviewer. Focus on regulatory "
                  "requirements and risk ratings.",
    allowed_tools=["risk_check", "approve_action"],
)

print("=== AgentRole Definitions ===")
for r in [role_analyst, role_reviewer]:
    d = r.to_dict()
    print(f"  {d['role_id']}: tools={d['allowed_tools']}")
    print(f"    prompt: '{d['system_prompt'][:60]}...'")

# Create mock agents
analyst = create_agent_for_role("analyst", expertise="credit risk")
reviewer = create_agent_for_role("reviewer", expertise="compliance")
print("\n=== Mock Agents ===")
print(f"  analyst:  role_id={analyst.role_id}, expertise={analyst.expertise}")
print(f"  reviewer: role_id={reviewer.role_id}, expertise={reviewer.expertise}")

In [ ]:
from importlib.resources import files
# Both agents answer the same question
ctx = open(str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md"))).read()
q = "What is the risk rating of MDL-001?"

ans_analyst = analyst.run(f"<report>{ctx}</report>\n\nAnswer: {q}")
ans_reviewer = reviewer.run(f"<report>{ctx}</report>\n\nAnswer: {q}")

print("=== Agent Outputs (same question) ===")
print(f"  Question: {q}")
print(f"  Analyst:  '{ans_analyst}'")
print(f"  Reviewer: '{ans_reviewer}'")
print(f"  Agree:    {ans_analyst == ans_reviewer}")

Both mock agents return `"0.8934"` — they agree trivially because
the mock always returns the same output. With real LLM-backed agents,
the analyst and reviewer might interpret "risk rating" differently
(quantitative AUC vs. categorical High/Medium/Low).

## ConsensusOracle: Inter-Agent Tension

The `ConsensusOracle` measures inter-agent consistency by computing
`tension_energy` across per-agent GMS stores.

In [7]:
# ConsensusOracle with same store (baseline — zero tension)
oracle_same = ConsensusOracle(
    stores={"analyst": store, "reviewer": store},
    tension_threshold=0.5,
)

entity = "MDL-001"
result_same = oracle_same.evaluate_consensus(entity=entity)
print("=== ConsensusOracle — same store (baseline) ===")
print(f"  Entity:           {entity}")
print(f"  Consensus:        {result_same.consensus_reached}")
print(f"  Mean tension:     {result_same.mean_tension:.4f}")
print(f"  Pairwise tension: {result_same.inter_agent_tension}")
print()
print("  (Same store => zero tension, consensus always reached)")

=== ConsensusOracle — same store (baseline) ===
  Entity:           MDL-001
  Consensus:        True
  Mean tension:     0.0000
  Pairwise tension: {}

  (Same store => zero tension, consensus always reached)


In [8]:
# ConsensusOracle with two different stores
oracle = ConsensusOracle(
    stores={"analyst": store, "reviewer": store2},
    tension_threshold=0.5,
)

entities_to_test = [
    "MDL-001", "MDL-002", "MDL-003",
    "MDL-005", "MDL-006", "CreditScore-XGB",
]

print("=== ConsensusOracle — two different stores ===")
print(f"  Tension threshold: {oracle.tension_threshold}")
print()
print(f"{'Entity':<25s} {'Tension':>10s} {'Consensus':>10s}")
print("-" * 50)
for ent in entities_to_test:
    r = oracle.evaluate_consensus(entity=ent)
    if r.inter_agent_tension:
        t_val = list(r.inter_agent_tension.values())[0]
        print(f"  {ent:<23s} {t_val:>10.4f} {str(r.consensus_reached):>10s}")
    else:
        print(f"  {ent:<23s}      {'N/A':>5s} {str(r.consensus_reached):>10s}")

print()
print("  Note: ConsensusOracle._cross_store_tension() uses")
print("  score_triple(entity, 'is_type', entity) which returns None")
print("  when 'is_type' is not in the store's relation vocabulary.")
print("  This is a known limitation — see manual scoring below.")

=== ConsensusOracle — two different stores ===
  Tension threshold: 0.5

Entity                       Tension  Consensus
--------------------------------------------------
  MDL-001                        N/A       True
  MDL-002                        N/A       True
  MDL-003                        N/A       True
  MDL-005                        N/A       True
  MDL-006                        N/A       True
  CreditScore-XGB                N/A       True

  Note: ConsensusOracle._cross_store_tension() uses
  score_triple(entity, 'is_type', entity) which returns None
  when 'is_type' is not in the store's relation vocabulary.
  This is a known limitation — see manual scoring below.


## Manual Cross-Store Triple Scoring

To demonstrate real cross-store tension, we score the *same* triples
across both stores using `score_triple()` with actual relation names
from the knowledge graph.

In [9]:
# Score the same triples across both stores
import numpy as np

triples = list(store.doc_graph.triples)
scored = []
for h, r, t in triples[:200]:
    s1 = store.score_triple(h, r, t)
    s2 = store2.score_triple(h, r, t)
    if s1 is not None and s2 is not None:
        diff = abs(float(s1) - float(s2))
        scored.append((h, r, t, float(s1), float(s2), diff))

# Sort by difference (descending)
scored.sort(key=lambda x: x[5], reverse=True)

print(f"=== Cross-Store Triple Scoring ({len(scored)} triples) ===")
print(f"{'Triple':<50s} {'Store1':>8s} {'Store2':>8s} {'|Diff|':>8s}")
print("-" * 80)
for h, r, t, s1, s2, diff in scored[:10]:
    label = f"({h}, {r}, {str(t)[:18]})"
    print(f"  {label:<48s} {s1:>8.4f} {s2:>8.4f} {diff:>8.4f}")

diffs = [x[5] for x in scored]
print("\n=== Geodesic Difference Statistics ===")
print(f"  Mean: {np.mean(diffs):.4f}")
print(f"  Std:  {np.std(diffs):.4f}")
print(f"  Min:  {np.min(diffs):.4f}")
print(f"  Max:  {np.max(diffs):.4f}")

=== Cross-Store Triple Scoring (200 triples) ===
Triple                                               Store1   Store2   |Diff|
--------------------------------------------------------------------------------
  (MDL-001, in_section, performance_metric)          1.2791   0.4016   0.8776
  (MDL-008, has_domain, Loss Given Default)          1.2177   1.6526   0.4350
  (MDL-005, has_training_date, 2025-10-10)           1.1728   1.5714   0.3987
  (MDL-003, in_section, performance_metric)          1.4397   1.0662   0.3735
  (MDL-001, in_section, model_inventory_su)          1.3356   0.9746   0.3610
  (MDL-002, has_train-test_gap, 0.0333)              1.5426   1.2004   0.3422
  (MDL-007, has_status, Stable)                      1.2198   1.5470   0.3272
  (MDL-002, has_tier, 1.0)                           1.2059   1.5293   0.3234
  (MDL-008, has_features, 67.0)                      1.2789   1.5978   0.3189
  (MDL-003, has_ks_statistic, 0.5234)                1.5365   1.2399   0.2966

=== Geodesi

In [10]:
# Cross-store tension energy for entity pairs
print("=== Cross-Store Tension Energy ===")
entity_pairs = [
    ("MDL-001", "MDL-002"), ("MDL-001", "MDL-003"),
    ("MDL-002", "FraudNet-LSTM"),
    ("CreditScore-XGB", "Credit Decisioning"),
    ("High", "Degraded"), ("MDL-005", "MDL-006"),
]
print(f"{'Pair':<45s} {'Store1':>8s} {'Store2':>8s} {'|Diff|':>8s}")
print("-" * 75)
for a, b in entity_pairs:
    t1 = store.tension_energy(a, b)
    t2 = store2.tension_energy(a, b)
    t1v = float(t1) if t1 is not None else None
    t2v = float(t2) if t2 is not None else None
    diff = abs(t1v - t2v) if t1v is not None and t2v is not None else None
    t1s = f"{t1v:.4f}" if t1v is not None else "None"
    t2s = f"{t2v:.4f}" if t2v is not None else "None"
    ds = f"{diff:.4f}" if diff is not None else "N/A"
    print(f"  ({a}, {b}){' '*(40-len(a)-len(b))} {t1s:>8s} {t2s:>8s} {ds:>8s}")

=== Cross-Store Tension Energy ===
Pair                                            Store1   Store2   |Diff|
---------------------------------------------------------------------------
  (MDL-001, MDL-002)                             1.4512   1.4479   0.0034
  (MDL-001, MDL-003)                             1.3830   1.4376   0.0546
  (MDL-002, FraudNet-LSTM)                       1.4299   1.4395   0.0096
  (CreditScore-XGB, Credit Decisioning)          1.5219   1.4376   0.0843
  (High, Degraded)                               1.4501   1.3613   0.0888
  (MDL-005, MDL-006)                             1.4112   1.3454   0.0658


## Delegation Holonomy

The `ConsensusOracle.evaluate_delegation_holonomy()` method checks
whether a delegation chain is geometrically consistent with a direct
relation. A holonomy defect of 0 means the chain is perfectly
consistent; higher values indicate geometric inconsistency.

In [11]:
# Delegation holonomy comparison
print("=== Delegation Holonomy ===")
test_chains = [
    (["has_risk_rating", "has_status"], "has_domain"),
    (["has_algorithm", "has_features"], "has_model_name"),
    (["has_tier", "has_risk_rating"], "has_status"),
]

print(f"{'Chain':<45s} {'Store1':>10s} {'Store2':>10s} {'|Diff|':>8s}")
print("-" * 78)
for chain, direct in test_chains:
    h1 = oracle.evaluate_delegation_holonomy(chain, direct, store)
    h2 = oracle.evaluate_delegation_holonomy(chain, direct, store2)
    h1v = float(h1) if h1 is not None else None
    h2v = float(h2) if h2 is not None else None
    diff = abs(h1v - h2v) if h1v is not None and h2v is not None else None
    h1s = f"{h1v:.4f}" if h1v is not None else "None"
    h2s = f"{h2v:.4f}" if h2v is not None else "None"
    ds = f"{diff:.4f}" if diff is not None else "N/A"
    label = f"{chain} -> {direct}"
    print(f"  {label:<43s} {h1s:>10s} {h2s:>10s} {ds:>8s}")

=== Delegation Holonomy ===
Chain                                             Store1     Store2   |Diff|
------------------------------------------------------------------------------
  ['has_risk_rating', 'has_status'] -> has_domain     2.5147     2.4970   0.0177
  ['has_algorithm', 'has_features'] -> has_model_name     2.5130     2.5338   0.0208
  ['has_tier', 'has_risk_rating'] -> has_status     2.5592     2.5254   0.0338


## Lab 19.1 — Multi-Agent Consensus Testing

Generate multi-agent coordination questions and run them through
mock agents with different roles.

In [12]:
# Lab 19.1a -- Generate multi-agent questions
gen = MultiAgentGenerator()
questions = gen.generate(store)

print("=== MultiAgentGenerator ===")
print(f"Generated {len(questions)} multi-agent questions\n")
print(f"{'QID':<20s} {'Entity':<20s} {'Type':<25s}")
print("-" * 68)
for q in questions[:8]:
    print(f"  {q.qid:<18s} {q.metadata['entity'][:18]:<18s} "
          f"{q.metadata['test_type']:<25s}")

=== MultiAgentGenerator ===
Generated 10 multi-agent questions

QID                  Entity               Type                     
--------------------------------------------------------------------
  multi_agent_000    0.7389             multi_agent_consistency  
  multi_agent_001    61.0               multi_agent_consistency  
  multi_agent_002    82.0               multi_agent_consistency  
  multi_agent_003    MDL-001            multi_agent_consistency  
  multi_agent_004    Critical           multi_agent_consistency  
  multi_agent_005    0.5123             multi_agent_consistency  
  multi_agent_006    Total Models in In multi_agent_consistency  
  multi_agent_007    2026-04-15         multi_agent_consistency  


In [13]:
# Lab 19.1b -- Run both agents on multi-agent questions
analyst = create_agent_for_role("analyst", expertise="credit risk")
reviewer = create_agent_for_role("reviewer", expertise="compliance")

print(f"{'QID':<20s} {'Analyst':<15s} {'Reviewer':<15s} {'Agree':>6s} {'GMS':>6s}")
print("-" * 68)
n_agree = 0
for q in questions[:10]:
    a_out = analyst.run(f"<report>{ctx}</report>\n\n{q.natural_language}")
    r_out = reviewer.run(f"<report>{ctx}</report>\n\n{q.natural_language}")
    agree = a_out == r_out
    gms_ans = q.gms_answer_fn(store)
    if agree:
        n_agree += 1
    print(f"  {q.qid:<18s} '{a_out[:11]}'  '{r_out[:11]}'  "
          f"{str(agree):>6s} {str(gms_ans):>6s}")

n_total = min(len(questions), 10)
print(f"\nAgreement rate: {n_agree}/{n_total} ({100*n_agree/n_total:.1f}%)")
print("\nNote: Mock agents always return '0.8934', so agreement is trivial.")
print("With real LLM agents, disagreement would trigger tension analysis.")

QID                  Analyst         Reviewer         Agree    GMS
--------------------------------------------------------------------


  multi_agent_000    '0.8934'  '0.8934'    True   True


  multi_agent_001    '0.8934'  '0.8934'    True   True


  multi_agent_002    '0.8934'  '0.8934'    True   True


  multi_agent_003    '0.8934'  '0.8934'    True   True
  multi_agent_004    '0.8934'  '0.8934'    True   True
  multi_agent_005    '0.8934'  '0.8934'    True   True


  multi_agent_006    '0.8934'  '0.8934'    True   True


  multi_agent_007    '0.8934'  '0.8934'    True   True
  multi_agent_008    '0.8934'  '0.8934'    True   True
  multi_agent_009    '0.8934'  '0.8934'    True   True

Agreement rate: 10/10 (100.0%)

Note: Mock agents always return '0.8934', so agreement is trivial.
With real LLM agents, disagreement would trigger tension analysis.


In [14]:
# Lab 19.1c -- InteractionResult construction
# Simulate what a real multi-agent orchestrator would produce
result = InteractionResult(
    scenario_id="model_review_consensus",
    agent_outputs={
        "analyst": "MDL-001 has test AUC 0.8412, rated High risk",
        "reviewer": "MDL-001 compliance status: Degraded, risk: High",
    },
    inter_agent_tension={
        "analyst:reviewer": 0.2435,
    },
    process_holonomy=1.2776,
    consensus_reached=True,
    mean_tension=0.2435,
)

print("=== InteractionResult (simulated) ===")
d = result.to_dict()
for k, v in d.items():
    print(f"  {k}: {v}")

=== InteractionResult (simulated) ===
  scenario_id: model_review_consensus
  n_agents: 2
  inter_agent_tension: {'analyst:reviewer': 0.2435}
  process_holonomy: 1.2776
  consensus_reached: True
  mean_tension: 0.2435


## Lab 19.2 — Cross-Document Conflict Detection

The `CrossDocumentAnalyzer` detects factual conflicts between
multiple document stores by scanning shared entities for
inter-store tension.

In [ ]:
# Lab 19.2a -- CrossDocumentAnalyzer: find shared entities
from knowlytix.harness.doe.cross_document import (
    CrossDocumentAnalyzer, CrossDocumentGenerator, DocumentConflict,
)

analyzer = CrossDocumentAnalyzer(
    stores={"primary": store, "secondary": store2},
    tension_threshold=0.1,
)

shared = analyzer.find_shared_entities()
print("=== Shared Entities ===")
print(f"  Total shared: {len(shared)}")
print(f"  Sample: {list(shared.keys())[:8]}")
print("  All entities are shared (same document, different training)")

In [ ]:
# Lab 19.2b -- Manual conflict detection via score_triple
# Since _compute_tension uses 'is_type' (not in vocabulary),
# we demonstrate conflict detection manually with real triples

from knowlytix.harness.doe.cross_document import _conflict_type

manual_conflicts = []
triples = list(store.doc_graph.triples)
for h, r, t in triples:
    s1 = store.score_triple(h, r, t)
    s2 = store2.score_triple(h, r, t)
    if s1 is not None and s2 is not None:
        tension = abs(float(s1) - float(s2))
        if tension >= 0.3:
            manual_conflicts.append(DocumentConflict(
                entity=h,
                store_a_id="primary",
                store_b_id="secondary",
                fact_a=f"{h} {r} {t}",
                fact_b=f"{h} {r} {t} (different embedding)",
                inter_store_tension=tension,
                conflict_type=_conflict_type(tension),
            ))

manual_conflicts.sort(key=lambda c: c.inter_store_tension, reverse=True)

print("=== Manual Cross-Store Conflicts (threshold >= 0.3) ===")
print(f"  Found {len(manual_conflicts)} conflicts\n")
print(f"{'Entity':<18s} {'Relation':<22s} {'Tension':>8s} {'Type':<16s}")
print("-" * 70)
for c in manual_conflicts[:10]:
    parts = c.fact_a.split(" ", 2)
    ent = parts[0] if parts else c.entity
    rel = parts[1][:20] if len(parts) > 1 else ""
    print(f"  {ent:<16s} {rel:<22s} {c.inter_store_tension:>8.4f} {c.conflict_type:<16s}")

# Summary by conflict type
from collections import Counter
type_counts = Counter(c.conflict_type for c in manual_conflicts)
print("\n=== Conflict Type Distribution ===")
for ct, n in sorted(type_counts.items()):
    print(f"  {ct:<18s}: {n}")

In [17]:
# Lab 19.2c -- Threshold sensitivity
print("=== Conflict Count by Threshold ===")
print(f"{'Threshold':>10s} {'Conflicts':>10s} {'Contradictions':>15s} {'Disagreements':>15s}")
print("-" * 55)
for threshold in [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    above = [c for c in manual_conflicts if c.inter_store_tension >= threshold]
    n_contra = sum(1 for c in above if c.conflict_type == "contradiction")
    n_disagr = sum(1 for c in above if c.conflict_type == "disagreement")
    print(f"  {threshold:>8.1f} {len(above):>10d} {n_contra:>15d} {n_disagr:>15d}")

=== Conflict Count by Threshold ===
 Threshold  Conflicts  Contradictions   Disagreements
-------------------------------------------------------
       0.1         29               0               3
       0.2         29               0               3
       0.3         29               0               3
       0.5          4               0               3
       0.7          3               0               3
       1.0          0               0               0


In [18]:
# Lab 19.2d -- CrossDocumentGenerator (intra-document mode)
gen_intra = CrossDocumentGenerator()  # no additional stores
intra_questions = gen_intra.generate(store)

print("=== CrossDocumentGenerator (intra-document) ===")
print(f"Generated {len(intra_questions)} questions\n")
if intra_questions:
    for q in intra_questions[:5]:
        print(f"  [{q.qid}] {q.natural_language[:90]}...")
        print(f"    tension={q.metadata.get('tension')}, type={q.metadata.get('test_type')}")
else:
    print("  No intra-document questions generated.")
    print("  This occurs when tension_energy() returns None for all")
    print("  entity pairs tested by the generator.")

=== CrossDocumentGenerator (intra-document) ===
Generated 0 questions

  No intra-document questions generated.
  This occurs when tension_energy() returns None for all
  entity pairs tested by the generator.


## End-to-End: Multi-Agent Consensus Pipeline

We combine all components: generate multi-agent questions, run
through role-specific agents, score cross-store geodesics and
measure holonomy for delegation chains.

In [19]:
# E2E: score all triples across both stores
all_scored = []
for h, r, t in store.doc_graph.triples:
    s1 = store.score_triple(h, r, t)
    s2 = store2.score_triple(h, r, t)
    if s1 is not None and s2 is not None:
        all_scored.append({
            "triple": (h, r, t),
            "geo_store1": float(s1),
            "geo_store2": float(s2),
            "diff": abs(float(s1) - float(s2)),
        })

print("=== E2E Cross-Store Scoring ===")
print(f"  Total triples: {len(list(store.doc_graph.triples))}")
print(f"  Scored in both stores: {len(all_scored)}")

diffs = [s["diff"] for s in all_scored]
print("\n=== Geodesic Difference Distribution ===")
print(f"  Mean: {np.mean(diffs):.4f}")
print(f"  Std:  {np.std(diffs):.4f}")
print(f"  Min:  {np.min(diffs):.4f}")
print(f"  Max:  {np.max(diffs):.4f}")

# Bucket by conflict type thresholds
n_minor = sum(1 for d in diffs if d < 0.7)
n_disagree = sum(1 for d in diffs if 0.7 <= d < 1.5)
n_contra = sum(1 for d in diffs if d >= 1.5)
total = len(diffs)
print("\n=== Conflict Classification ===")
print(f"  minor_variance  (<0.7):   {n_minor:>5d} ({100*n_minor/total:.1f}%)")
print(f"  disagreement  (0.7-1.5):  {n_disagree:>5d} ({100*n_disagree/total:.1f}%)")
print(f"  contradiction (>=1.5):    {n_contra:>5d} ({100*n_contra/total:.1f}%)")

=== E2E Cross-Store Scoring ===
  Total triples: 807
  Scored in both stores: 807

=== Geodesic Difference Distribution ===
  Mean: 0.1174
  Std:  0.0972
  Min:  0.0003
  Max:  0.9345

=== Conflict Classification ===
  minor_variance  (<0.7):     804 (99.6%)
  disagreement  (0.7-1.5):      3 (0.4%)
  contradiction (>=1.5):        0 (0.0%)


In [20]:
# E2E: holonomy comparison for multiple delegation chains
rels = list(store.adapter.relation_to_idx.keys())
chain_tests = [
    (["has_risk_rating", "has_status"], "has_domain"),
    (["has_algorithm", "has_features"], "has_model_name"),
    (["has_tier", "has_risk_rating"], "has_status"),
    (["has_domain", "has_algorithm"], "has_tier"),
    (["has_status", "has_tier"], "has_risk_rating"),
]

print("=== E2E Holonomy Comparison ===")
print(f"{'Chain -> Direct':<48s} {'Store1':>8s} {'Store2':>8s} {'|Diff|':>8s}")
print("-" * 78)
holonomy_diffs = []
for chain, direct in chain_tests:
    h1 = store.check_holonomy(chain, direct)
    h2 = store2.check_holonomy(chain, direct)
    h1v = float(h1) if h1 is not None else None
    h2v = float(h2) if h2 is not None else None
    diff = abs(h1v - h2v) if h1v is not None and h2v is not None else None
    h1s = f"{h1v:.4f}" if h1v is not None else "None"
    h2s = f"{h2v:.4f}" if h2v is not None else "None"
    ds = f"{diff:.4f}" if diff is not None else "N/A"
    label = f"{chain} -> {direct}"
    print(f"  {label:<46s} {h1s:>8s} {h2s:>8s} {ds:>8s}")
    if diff is not None:
        holonomy_diffs.append(diff)

if holonomy_diffs:
    print(f"\n  Mean holonomy difference: {np.mean(holonomy_diffs):.4f}")
    print(f"  Max holonomy difference:  {np.max(holonomy_diffs):.4f}")

=== E2E Holonomy Comparison ===
Chain -> Direct                                    Store1   Store2   |Diff|
------------------------------------------------------------------------------
  ['has_risk_rating', 'has_status'] -> has_domain   2.5147   2.4970   0.0177
  ['has_algorithm', 'has_features'] -> has_model_name   2.5130   2.5338   0.0208
  ['has_tier', 'has_risk_rating'] -> has_status    2.5592   2.5254   0.0338
  ['has_domain', 'has_algorithm'] -> has_tier      2.5349   2.4939   0.0410
  ['has_status', 'has_tier'] -> has_risk_rating    2.5237   2.5073   0.0164

  Mean holonomy difference: 0.0260
  Max holonomy difference:  0.0410


In [21]:
# Summary statistics
print("=== Chapter 19 Summary ===")
print(f"  Store 1: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples (200 epochs)")
print(f"  Store 2: {store2.stats()['entities']} entities, "
      f"{store2.stats()['triples']} triples (100 epochs)")
print(f"  Multi-agent questions generated: {len(questions)}")
print(f"  Mock agent agreement rate: {n_agree}/{n_total} ({100*n_agree/n_total:.1f}%)")
print(f"  Cross-store triples scored: {len(all_scored)}")
print(f"  Mean geodesic difference: {np.mean(diffs):.4f}")
print(f"  Manual conflicts (>=0.3): {len(manual_conflicts)}")
print(f"  Holonomy chains compared: {len(chain_tests)}")
print(f"  Mean holonomy difference: {np.mean(holonomy_diffs):.4f}")

=== Chapter 19 Summary ===
  Store 1: 508 entities, 807 triples (200 epochs)
  Store 2: 508 entities, 807 triples (100 epochs)
  Multi-agent questions generated: 10
  Mock agent agreement rate: 10/10 (100.0%)
  Cross-store triples scored: 807
  Mean geodesic difference: 0.1174
  Manual conflicts (>=0.3): 29
  Holonomy chains compared: 5
  Mean holonomy difference: 0.0260


## Exercises

1. **Exercise 19.1.** Create a third store by ingesting the same document
   with `ingest_epochs=50`. Add it to the `ConsensusOracle` as a third
   agent ("approver"). Score the same triples across all three stores
   and compute the pairwise geodesic differences. Does the lowest-epoch
   store consistently disagree more with the 200-epoch store?

2. **Exercise 19.2.** Build a `DocumentConflict` detector that uses
   `tension_energy()` (instead of `score_triple()`) between entity pairs.
   For each shared entity, compute `tension_energy(entity, entity)` in
   both stores and flag entities where the inter-store difference exceeds
   0.2. Compare the conflict count and type distribution with the
   `score_triple`-based approach.

3. **Exercise 19.3.** Write a 3-agent consensus script: analyst generates
   a risk assessment, reviewer checks compliance, approver makes the
   final decision. Use `InteractionScript` from Chapter 18 to chain the
   three agents. For each shared entity mentioned, compute the pairwise
   holonomy defect for the delegation chain
   `["analyst_assesses", "reviewer_validates"]` vs. `"approver_decides"`.
   What does the holonomy tell you about the delegation's consistency?

## Summary

This chapter demonstrated **multi-agent consensus** and **cross-document
consistency** testing using GMS geometric primitives:

- **AgentRole** and `create_agent_for_role()` define and instantiate
  role-specific agents for multi-agent testing.
- **ConsensusOracle** measures inter-agent tension via `tension_energy()`
  across per-agent stores.
- **Delegation holonomy** checks whether multi-step agent workflows
  are geometrically consistent with direct relations.
- **Cross-store triple scoring** reveals embedding divergence between
  stores trained with different hyperparameters.
- **CrossDocumentAnalyzer** detects factual conflicts by classifying
  inter-store tension into minor variance, disagreement and contradiction.